# rlmflow visualization walkthrough

A guided tour of every visualization that ships with rlmflow. Everything here renders **inline in Jupyter** and runs **offline** against the saved boids trace under `examples/data/notebook-coding-agent/` — no API keys, no LLM calls.

The trace is regenerated by `examples/data/build_notebook_fixture.py`. For the underlying `Graph` query API (walk, find, where, session), see [`node_basics.ipynb`](./node_basics.ipynb).

**What we cover**

1. Loading a trace
2. Inline tree rendering — `graph.tree()`, `ascii_boxes`
3. **Interactive viewer** — `open_viewer` (Gradio: clickable graph, slider, per-agent chat)
4. Plotly graph + HTML stepper — `graph.plot()`, `render_html`, `save_steps`
5. Topology renders — Mermaid (state / flowchart / sequence), DOT, D2
6. Step-indexed timeline — `gantt`, `gantt_html`
7. Per-state detail — `code_log`, `error_summary`, `message_stream`, `diff_system_prompts`
8. Cost & reports — `token_sparkline`, `budget_burndown`, `report_md`
9. Comparison across runs — `bench_table`
10. CLI equivalents

## 1. Load a trace

`load_trace` returns a `Trace` with `graphs` (one snapshot per step) and `metadata`. Everything below is a function over a `Graph` or `list[Graph]`.

In [ ]:
from rlmflow.utils.trace import load_trace

trace = load_trace("../data/notebook-coding-agent/trace")
graphs = trace.graphs
final = graphs[-1]
print(f"loaded {len(graphs)} snapshots  \u00b7  metadata: {trace.metadata or '(none)'}")

## 2. Inline tree rendering

`graph.tree()` returns a per-agent timeline as plain text. `ascii_boxes` is a richer variant that wraps each agent in a Rich panel — better for pasting into PR comments or terminal screenshots.

In [ ]:
print(final.tree())

In [ ]:
from rlmflow.utils.viz import ascii_boxes

print(ascii_boxes(final))

## 3. Interactive viewer

The full interactive graph + per-agent chat lives in `open_viewer`. It embeds inline in Jupyter via Gradio, with a step slider, clickable nodes, and a color-coded conversation panel for any agent you click.

For static export (PR comments, email), use the Plotly stepper in section 4 or the topology renders in section 5.

In [ ]:
from rlmflow.utils.viewer import open_viewer

open_viewer(
    graphs,
    inline=True,
    height=720,
    quiet=True,
)

## 4. Plotly graph + HTML stepper

`graph.plot()` builds a Plotly figure with every node laid out as a tidy tree (one column per branching child, edges from `Graph.edges`). `render_html(graphs)` stitches every snapshot into a self-contained HTML page with arrow-key navigation; `save_steps(graphs, out_dir)` writes one PNG per step (handy for slide decks).

In [ ]:
final.plot(height=500)

In [ ]:
import tempfile
from pathlib import Path
from IPython.display import IFrame
from rlmflow.utils.viewer import save_html

out = Path(tempfile.gettempdir()) / "rlmflow-stepper.html"
save_html(graphs, out)
print(f"wrote {out}")
IFrame(str(out), width="100%", height=720)

## 5. Topology renders

Static topology renders for embedding in docs, PRs, post-mortems. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are reachable from the CLI as `rlmflow render <path> -f F`.

In [ ]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(final))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(final))

## 6. Step-indexed timeline

`gantt(graphs)` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html(graphs)` writes a colorful self-contained HTML page; we render it inline below.

In [ ]:
from rlmflow.utils.viz import gantt

gantt(graphs)

In [ ]:
from IPython.display import HTML
from rlmflow.utils.viz import gantt_html

HTML(gantt_html(graphs, title="boids run"))

## 7. Per-state detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent.

In [ ]:
from rlmflow.utils.viz import code_log

print(code_log(final))

In [ ]:
# The boids run had no errors — synthesize a tiny graph so error_summary has something to group.
from rlmflow.graph import ErrorNode, Graph, QueryNode
from rlmflow.utils.viz import error_summary

root_q = QueryNode(agent_id="root", seq=0, content="demo")
def child(aid: str, err: str, msg: str) -> Graph:
    return Graph(
        agent_id=aid, depth=1, parent_agent_id="root", parent_node_id=root_q.id,
        states=(ErrorNode(agent_id=aid, seq=0, error=err, content=msg),),
    )

demo = Graph(
    agent_id="root",
    states=(root_q,),
    children={
        "root.a": child("root.a", "no_code_block", "missing repl block"),
        "root.b": child("root.b", "no_code_block", "empty reply"),
        "root.c": child("root.c", "orphaned_delegates", "never waited"),
    },
)
print(error_summary(demo))

In [ ]:
from rlmflow.utils.viz import message_stream

stream = message_stream("root.sim_js", final)
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

In [ ]:
from rlmflow.utils.viz import diff_system_prompts

children = sorted(aid for aid in final if aid != "root")
print("child agents:", children)
a, b = children[0], children[1]
print(f"\ndiffing: {a}  vs  {b}\n")
diff = diff_system_prompts(final, final, a)  # same graph, so prompts match
print(diff)

## 8. Cost & reports

Three views: a one-line ASCII sparkline of cumulative tokens, a budget burndown bar, and a full Markdown report bundling tree + cost + result + errors.

In [ ]:
from rlmflow.utils.viz import budget_burndown, token_sparkline

print(token_sparkline(graphs))
print(budget_burndown(graphs))
print(budget_burndown(graphs, max_budget=50_000))

In [ ]:
from rlmflow.utils.viz import report_md

Markdown(report_md(graphs, title="boids \u2014 coding-agent run"))

## 9. Comparison across runs

`bench_table` aggregates labeled traces — use it to compare benchmark runs side by side (cost, duration, outcome, errors).

In [ ]:
from rlmflow.utils.viz import bench_table

# Fake two runs by trimming the same trace, just to show the table shape.
print(
    bench_table(
        {
            "boids:full": graphs,
            "boids:firsthalf": graphs[: len(graphs) // 2 + 1],
        }
    )
)

## 10. CLI equivalents

Every render here is also available without writing Python. From a shell:

```bash
rlmflow view   examples/data/notebook-coding-agent/trace
rlmflow render examples/data/notebook-coding-agent/trace -f mermaid-flowchart
rlmflow render examples/data/notebook-coding-agent/trace -f gantt-html  -o gantt.html
rlmflow render examples/data/notebook-coding-agent/trace -f report-md   -o report.md
rlmflow render examples/data/notebook-coding-agent/trace -f code-log
rlmflow render examples/data/notebook-coding-agent/trace -f tokens
```

All formats: `mermaid` \u00b7 `mermaid-flowchart` \u00b7 `mermaid-sequence` \u00b7 `dot` \u00b7 `d2` \u00b7 `tree` \u00b7 `ascii-boxes` \u00b7 `gantt-html` \u00b7 `report-md` \u00b7 `code-log` \u00b7 `error-summary` \u00b7 `tokens`.